In [1]:
"""
PR3 unit tests — MoiraicForecast end-to-end with use_cache.

Reference is a hand-rolled, no-cache AR forecast that ALSO does not flip
prediction_mask on commits ("appends-True" semantics). This is the only
fair comparison for the cache, since the existing use_cache=False path
flips prediction_mask and therefore feeds AR predictions into the scaler.
"""
import math
import torch
from einops import rearrange, repeat

from uni2ts.model.moiraic.forecast import MoiraicForecast
from uni2ts.model.moiraic.module import MoiraicModule

torch.manual_seed(0)
device = "cuda:7" if torch.cuda.is_available() else "cpu"
ATOL = 1e-4


def make_forecast(prediction_length, context_length, num_predict_token=1,
                  patch_size=8, target_dim=1, use_cache=True):
    module = MoiraicModule(
        d_model=64, d_ff=128, num_layers=2,
        patch_size=patch_size, max_seq_len=128,
        attn_dropout_p=0.0, dropout_p=0.0,
        scaling=True, num_predict_token=num_predict_token,
    ).to(device).eval()
    forecast = MoiraicForecast(
        prediction_length=prediction_length,
        target_dim=target_dim,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
        context_length=context_length,
        module=module,
        use_cache=use_cache,
    ).to(device).eval()
    return forecast


def make_inputs(B, context_length, target_dim=1):
    past_target = torch.randn(B, context_length, target_dim, device=device)
    past_observed = torch.ones(B, context_length, target_dim, dtype=torch.bool, device=device)
    past_is_pad = torch.zeros(B, context_length, dtype=torch.bool, device=device)
    return past_target, past_observed, past_is_pad


# ---------------------------------------------------------------------------
# Reference: no-cache, frozen-pmask AR forecast.
# Mirrors MoiraicForecast.forward(use_cache=False) but skips the
# `expand_prediction_mask[..., adjusted_assign_index] = False` line.
# ---------------------------------------------------------------------------
@torch.no_grad()
def reference_forward_frozen_pmask(forecast, past_target, past_observed, past_is_pad):
    target, observed_mask, sample_id, time_id, variate_id, prediction_mask = forecast._convert(
        forecast.module.patch_size, past_target, past_observed, past_is_pad,
    )
    per_var_ctx = forecast.context_token_length(forecast.module.patch_size)
    total_ctx = forecast.hparams.target_dim * per_var_ctx
    per_var_pred = forecast.prediction_token_length(forecast.module.patch_size)
    total_pred = forecast.hparams.target_dim * per_var_pred

    pred_index = torch.arange(per_var_ctx - 1, total_ctx, per_var_ctx)
    assign_index = torch.arange(total_ctx, total_ctx + total_pred, per_var_pred)

    quantile_prediction = repeat(
        target, "... patch -> ... q patch",
        q=len(forecast.module.quantile_levels),
        patch=forecast.module.patch_size,
    ).clone()

    preds = forecast.module(
        target, observed_mask, sample_id, time_id, variate_id, prediction_mask,
        training_mode=False,
    )

    def smp(per_var_pred, pred_index, assign_index, preds):
        preds = rearrange(
            preds,
            "... (pt q p) -> ... pt q p",
            pt=forecast.module.num_predict_token,
            q=forecast.module.num_quantiles,
            p=forecast.module.patch_size,
        )
        preds = rearrange(
            preds[..., pred_index, :per_var_pred, :, :],
            "... pi pt q p -> ... (pi pt) q p",
        )
        adj = torch.cat([torch.arange(i, i + per_var_pred) for i in assign_index])
        return preds, adj

    if per_var_pred <= forecast.module.num_predict_token:
        preds, adj = smp(per_var_pred, pred_index, assign_index, preds)
        quantile_prediction[..., adj, :, :] = preds
        return forecast._format_preds(
            forecast.module.num_quantiles, forecast.module.patch_size,
            quantile_prediction, forecast.hparams.target_dim,
        )

    # Expand by num_quantiles
    Q = len(forecast.module.quantile_levels)
    et = repeat(target, "b ... -> b q ...", q=Q).clone()
    epm = repeat(prediction_mask, "b ... -> b q ...", q=Q).clone()
    eom = repeat(observed_mask, "b ... -> b q ...", q=Q).clone()
    esi = repeat(sample_id, "b ... -> b q ...", q=Q).clone()
    eti = repeat(time_id, "b ... -> b q ...", q=Q).clone()
    evi = repeat(variate_id, "b ... -> b q ...", q=Q).clone()

    preds, adj = smp(forecast.module.num_predict_token, pred_index, assign_index, preds)
    quantile_prediction[..., adj, :, :] = preds
    et[..., adj, :] = rearrange(
        preds, "... pt q p -> ... q pt p",
        q=forecast.module.num_quantiles,
        p=forecast.module.patch_size,
        pt=forecast.module.num_predict_token,
    )
    # FROZEN-PMASK: do NOT flip epm[..., adj] = False

    remain = per_var_pred - forecast.module.num_predict_token
    while remain > 0:
        preds = forecast.module(et, eom, esi, eti, evi, epm, training_mode=False)
        pred_index = assign_index + forecast.module.num_predict_token - 1
        assign_index = pred_index + 1
        preds, adj = smp(
            forecast.module.num_predict_token if remain - forecast.module.num_predict_token > 0 else remain,
            pred_index, assign_index, preds,
        )
        qns = rearrange(
            preds, "... qp pi q p -> ... pi (qp q) p",
            q=forecast.module.num_quantiles, p=forecast.module.patch_size,
        )
        qns = torch.quantile(
            qns,
            torch.tensor(forecast.module.quantile_levels, device=device, dtype=torch.float32),
            dim=-2,
        )
        quantile_prediction[..., adj, :, :] = rearrange(qns, "q ... p -> ... q p")
        et[..., adj, :] = rearrange(
            qns, "q b pt p -> b q pt p",
            q=forecast.module.num_quantiles,
            p=forecast.module.patch_size,
            pt=len(adj),
        )
        # FROZEN-PMASK: do NOT flip epm
        remain -= forecast.module.num_predict_token

    return forecast._format_preds(
        forecast.module.num_quantiles, forecast.module.patch_size,
        quantile_prediction, forecast.hparams.target_dim,
    )

In [2]:
# ---------------------------------------------------------------------------
# T1 — Single-shot path (per_var_predict_token <= num_predict_token):
#      cache is irrelevant, output equals use_cache=False path.
# ---------------------------------------------------------------------------
def test_single_shot_path_cache_is_noop():
    # patch_size=8, prediction_length=8 -> per_var_predict_token=1 <= num_predict_token=1
    fc_cached = make_forecast(prediction_length=8, context_length=40, num_predict_token=1)
    fc_uncached = make_forecast(prediction_length=8, context_length=40, num_predict_token=1, use_cache=False)
    # Force the same module weights
    fc_uncached.module.load_state_dict(fc_cached.module.state_dict())

    pt, po, pp = make_inputs(B=2, context_length=40)
    with torch.no_grad():
        out_c = fc_cached(pt, po, pp)
        out_u = fc_uncached(pt, po, pp)
    diff = (out_c - out_u).abs().max().item()
    assert diff < ATOL, f"single-shot mismatch: {diff:.2e}"
    print("✓ test_single_shot_path_cache_is_noop")


test_single_shot_path_cache_is_noop()

✓ test_single_shot_path_cache_is_noop


In [3]:
# ---------------------------------------------------------------------------
# T2 — AR path, single variate: cached forecast equals frozen-pmask reference.
# ---------------------------------------------------------------------------
def test_ar_cached_matches_frozen_pmask_reference_univariate():
    # patch_size=8, prediction_length=32, num_predict_token=1
    # -> per_var_predict_token=4, AR loop runs 3 extra steps
    fc = make_forecast(prediction_length=32, context_length=40, num_predict_token=1, target_dim=1)
    pt, po, pp = make_inputs(B=2, context_length=40, target_dim=1)
    with torch.no_grad():
        out_cached = fc(pt, po, pp)
        out_ref = reference_forward_frozen_pmask(fc, pt, po, pp)
    diff = (out_cached - out_ref).abs().max().item()
    assert diff < ATOL, f"univariate AR cached vs frozen-pmask ref mismatch: {diff:.2e}"
    print(f"✓ test_ar_cached_matches_frozen_pmask_reference_univariate (max diff {diff:.2e})")


test_ar_cached_matches_frozen_pmask_reference_univariate()


✓ test_ar_cached_matches_frozen_pmask_reference_univariate (max diff 0.00e+00)


In [4]:
# ---------------------------------------------------------------------------
# T3 — AR path, multivariate: cached forecast equals frozen-pmask reference.
# ---------------------------------------------------------------------------
def test_ar_cached_matches_frozen_pmask_reference_multivariate():
    fc = make_forecast(prediction_length=32, context_length=40, num_predict_token=1, target_dim=2)
    pt, po, pp = make_inputs(B=2, context_length=40, target_dim=2)
    with torch.no_grad():
        out_cached = fc(pt, po, pp)
        out_ref = reference_forward_frozen_pmask(fc, pt, po, pp)
    diff = (out_cached - out_ref).abs().max().item()
    assert diff < ATOL, f"multivariate AR cached vs frozen-pmask ref mismatch: {diff:.2e}"
    print(f"✓ test_ar_cached_matches_frozen_pmask_reference_multivariate (max diff {diff:.2e})")


# test_ar_cached_matches_frozen_pmask_reference_multivariate()
print("Didn't run because Moiraic doesn't allow for multivariate forecasting")

Didn't run because Moiraic doesn't allow for multivariate forecasting


In [10]:
# ---------------------------------------------------------------------------
# T4 — Toggle: setting forecast.use_cache=False at runtime changes outputs
#       to match the existing flip-pmask path (and is generally != cached).
# ---------------------------------------------------------------------------
def test_use_cache_flag_toggles_behavior():
    fc = make_forecast(prediction_length=32, context_length=40, num_predict_token=1)
    pt, po, pp = make_inputs(B=2, context_length=40)
    with torch.no_grad():
        out_cached = fc(pt, po, pp)
        fc.use_cache = False
        fc.update_context_in_ar = True
        out_flip = fc(pt, po, pp)
        fc.use_cache = True
        fc.update_context_in_ar = False
        out_cached_again = fc(pt, po, pp)

    # Cached call is reproducible
    assert torch.allclose(out_cached, out_cached_again, atol=ATOL)
    # And differs from the flip-pmask path (different scaler semantics by design).
    diff = (out_cached - out_flip).abs().max().item()
    assert diff > 1e-3, (
        f"Expected divergence between cached (frozen-pmask) and "
        f"use_cache=False (flip-pmask) paths, got {diff:.2e}. "
        "Either the cache is silently flipping pmask, or the model is "
        "insensitive to scaler updates on this synthetic input — try a "
        "longer prediction horizon."
    )
    print(f"✓ test_use_cache_flag_toggles_behavior (cached vs flip diff {diff:.2e})")


test_use_cache_flag_toggles_behavior()

✓ test_use_cache_flag_toggles_behavior (cached vs flip diff 3.38e-01)


In [6]:
# ---------------------------------------------------------------------------
# T5 — Default flag value is True.
# ---------------------------------------------------------------------------
def test_default_use_cache_is_true():
    module = MoiraicModule(
        d_model=64, d_ff=128, num_layers=2, patch_size=8, max_seq_len=128,
        attn_dropout_p=0.0, dropout_p=0.0, scaling=True, num_predict_token=1,
    ).to(device).eval()
    fc = MoiraicForecast(
        prediction_length=16, target_dim=1, feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0, context_length=24, module=module,
    ).to(device).eval()
    assert fc.use_cache is True
    print("✓ test_default_use_cache_is_true")


test_default_use_cache_is_true()

✓ test_default_use_cache_is_true


In [7]:
# ---------------------------------------------------------------------------
# T6 — Multi-step AR with num_predict_token>1: still matches reference.
# ---------------------------------------------------------------------------
def test_ar_with_multi_predict_token():
    # num_predict_token=2 means each AR step commits 2 patches per variate.
    fc = make_forecast(prediction_length=48, context_length=40, num_predict_token=2)
    pt, po, pp = make_inputs(B=2, context_length=40)
    with torch.no_grad():
        out_cached = fc(pt, po, pp)
        out_ref = reference_forward_frozen_pmask(fc, pt, po, pp)
    diff = (out_cached - out_ref).abs().max().item()
    assert diff < ATOL, f"multi-predict-token AR mismatch: {diff:.2e}"
    print(f"✓ test_ar_with_multi_predict_token (max diff {diff:.2e})")


test_ar_with_multi_predict_token()

✓ test_ar_with_multi_predict_token (max diff 0.00e+00)


In [8]:
# ---------------------------------------------------------------------------
# T7 — update_context_in_ar=False (no cache) matches use_cache=True (cache).
#       Both implement frozen-scaler AR semantics; the cache is just faster.
# ---------------------------------------------------------------------------
def test_update_context_flag_uncached_matches_cached():
    # Univariate, AR active (per_var_predict_token > num_predict_token).
    # patch_size=8, prediction_length=32, num_predict_token=1 -> 4 AR steps.
    fc_cached = make_forecast(
        prediction_length=32, context_length=40, num_predict_token=1,
        target_dim=1, use_cache=True,
    )
    fc_uncached_frozen = MoiraicForecast(
        prediction_length=32, target_dim=1, feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0, context_length=40,
        module=fc_cached.module,                     # share weights
        use_cache=False,
        update_context_in_ar=False,                  # <-- the new path
    ).to(device).eval()
    fc_uncached_default = MoiraicForecast(
        prediction_length=32, target_dim=1, feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0, context_length=40,
        module=fc_cached.module,                     # share weights
        use_cache=False,                             # default update_context_in_ar=True
    ).to(device).eval()

    pt, po, pp = make_inputs(B=2, context_length=40, target_dim=1)
    with torch.no_grad():
        out_cached = fc_cached(pt, po, pp)
        out_uncached_frozen = fc_uncached_frozen(pt, po, pp)
        out_uncached_default = fc_uncached_default(pt, po, pp)

    # Cached and uncached-frozen must match: same semantics, two implementations.
    diff_match = (out_cached - out_uncached_frozen).abs().max().item()
    assert diff_match < ATOL, (
        f"cached vs uncached-frozen mismatch: {diff_match:.2e}. "
        "These should implement identical semantics."
    )

    # Cached and uncached-default must DIFFER (different scaler semantics).
    diff_split = (out_cached - out_uncached_default).abs().max().item()
    assert diff_split > 1e-3, (
        f"Expected divergence between cached (frozen) and uncached-default "
        f"(updates context) paths, got {diff_split:.2e}. Either the toggle is "
        "not wired correctly, or the inputs don't trigger meaningful scaler drift."
    )
    print(
        f"✓ test_update_context_flag_uncached_matches_cached "
        f"(cached==frozen diff {diff_match:.2e}, vs default diff {diff_split:.2e})"
    )


test_update_context_flag_uncached_matches_cached()

✓ test_update_context_flag_uncached_matches_cached (cached==frozen diff 0.00e+00, vs default diff 2.76e-01)


In [9]:
# ---------------------------------------------------------------------------
# T8 — Default flag values: use_cache=True, update_context_in_ar derived as False.
# ---------------------------------------------------------------------------
def test_default_update_context_flag_derivation():
    module = MoiraicModule(
        d_model=64, d_ff=128, num_layers=2, patch_size=8, max_seq_len=128,
        attn_dropout_p=0.0, dropout_p=0.0, scaling=True, num_predict_token=1,
    ).to(device).eval()

    fc_default = MoiraicForecast(
        prediction_length=16, target_dim=1, feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0, context_length=24, module=module,
    )
    assert fc_default.use_cache is True
    assert fc_default.update_context_in_ar is False  # derived from use_cache

    fc_uncached = MoiraicForecast(
        prediction_length=16, target_dim=1, feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0, context_length=24, module=module,
        use_cache=False,
    )
    assert fc_uncached.use_cache is False
    assert fc_uncached.update_context_in_ar is True  # derived: preserve old behavior

    # Explicit override beats derivation.
    fc_explicit = MoiraicForecast(
        prediction_length=16, target_dim=1, feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0, context_length=24, module=module,
        use_cache=False, update_context_in_ar=False,
    )
    assert fc_explicit.update_context_in_ar is False
    print("✓ test_default_update_context_flag_derivation")


test_default_update_context_flag_derivation()

✓ test_default_update_context_flag_derivation
